# NB10 — MDPI Final Publication Figures

**Project:** `DRY_BEAN_HYBRID_Q1`  
**Purpose:** regenerate the eight manuscript figures from the final saved analytical outputs using a consistent publication-ready visual system.

### MDPI figure policy used in this notebook
- **No figure title is embedded inside any image.**
- **No descriptive subtitle/caption is embedded inside any image.**
- Figure names and explanations belong only in the manuscript caption.
- Inside each plot, only scientifically necessary elements are retained: axes, tick labels, legends, panel labels, values, and color bars.
- An automatic quality-control check stops export if a Matplotlib title, suptitle, or figure-level text is detected.

### Runtime
**CPU only.** No model is retrained in this notebook. It reuses final outputs produced by the previous notebooks.

### Outputs
Figures are saved to:

`/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1/05_FIGURES/NB10_PUBLICATION_FIGURES_MDPI_CLEAN`

Each figure is exported as PNG (600 dpi), PDF, and SVG.


In [ ]:

# ============================================================
# 0. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted.")
print("Runtime recommendation: CPU only.")


In [ ]:

# ============================================================
# 1. Imports and project paths
# ============================================================
from pathlib import Path
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.optimize import minimize_scalar
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, roc_curve, auc

warnings.filterwarnings("ignore")

ROOT = Path("/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1")
DATA = ROOT / "01_DATA"
RESULTS = ROOT / "03_RESULTS"
FIGURES = ROOT / "05_FIGURES"
OUT = FIGURES / "NB10_PUBLICATION_FIGURES_MDPI_FINAL_R13"
OUT.mkdir(parents=True, exist_ok=True)

PATHS = {
    "dataset": DATA / "INIAP_Dataset.xlsx",
    "hybrid_oof": RESULTS / "NB04_HYBRIDS" / "hybrid_oof_predictions.csv",
    "learning": RESULTS / "NB07_ROC_LEARNING" / "learning_curve_by_fold.csv",
    "tabpfn_oof": RESULTS / "NB08_EXTENDED_ABLATION" / "FULL_11_TabPFN_oof.csv",
    "metrics": RESULTS / "NB09_ROBUSTNESS_CALIBRATION_SELECTIVE" / "all_metrics_harmonized.csv",
    "selective": RESULTS / "NB09_ROBUSTNESS_CALIBRATION_SELECTIVE" / "selective_classification_by_seed.csv",
}

print("ROOT:", ROOT)
print("OUT :", OUT)


In [ ]:

# ============================================================
# 2. Validate required files
# ============================================================
missing = [f"{k}: {p}" for k, p in PATHS.items() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required input file(s):\n\n" + "\n".join(missing)
    )

print("All required files were found:")
for k, p in PATHS.items():
    print(f"  {k:12s} -> {p}")


In [ ]:

# ============================================================
# 3. Load final analytical outputs
# ============================================================
raw = pd.read_excel(PATHS["dataset"])
metrics = pd.read_csv(PATHS["metrics"])
hybrid_oof = pd.read_csv(PATHS["hybrid_oof"])
tabpfn_oof = pd.read_csv(PATHS["tabpfn_oof"])
selective = pd.read_csv(PATHS["selective"])
learning = pd.read_csv(PATHS["learning"])

# Raw dataset contains the historical field name "AspectRation".
# Rename in memory only; the source file is never modified.
df = raw.rename(columns={"AspectRation": "AspectRatio"}).copy()

TARGET = "Class"
FEATURES = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "ConvexArea", "EquivDiameter", "Extent",
    "Solidity", "roundness", "Compactness"
]
CLASS_ORDER = ["INIAP 420", "INIAP 425", "INIAP 481", "INIAP 485"]
PROB_COLS = [f"prob_{c}" for c in CLASS_ORDER]

print("Dataset shape:", df.shape)
print("Classes:", df[TARGET].value_counts().to_dict())
print("Metrics rows:", len(metrics))
print("Hybrid OOF rows:", len(hybrid_oof))
print("TabPFN OOF rows:", len(tabpfn_oof))
print("Selective rows:", len(selective))
print("Learning-curve rows:", len(learning))


In [ ]:

# ============================================================
# 4. Publication visual system
# ============================================================
COLORS = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "black": "#222222",
    "gray": "#7A7A7A",
}

CLASS_COLORS = {
    "INIAP 420": COLORS["blue"],
    "INIAP 425": COLORS["orange"],
    "INIAP 481": COLORS["green"],
    "INIAP 485": COLORS["vermillion"],
}

MODEL_COLORS = {
    "TabPFN": COLORS["vermillion"],
    "LDA-XGBoost": COLORS["blue"],
    "LDA-SVM": COLORS["green"],
    "SVM-RBF": COLORS["orange"],
    "FT-Transformer": COLORS["purple"],
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10.5,
    "axes.labelsize": 11,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9.5,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

manifest = []

def clean_model_name(name):
    mapping = {
        "LDA_XGBoost": "LDA-XGBoost",
        "LDA_SVM": "LDA-SVM",
        "SVM_RBF": "SVM-RBF",
        "FTTransformer": "FT-Transformer",
        "RandomForest": "Random Forest",
        "LDA_Shrinkage": "Shrinkage LDA",
        "PCA_XGBoost": "PCA-XGBoost",
        "PCA_SVM": "PCA-SVM",
        "PCA_MLP": "PCA-MLP",
        "LDA_MLP": "LDA-MLP",
        "LogisticRegression": "Logistic regression",
        "KNN": "k-NN",
    }
    return mapping.get(str(name), str(name).replace("_", "-"))

def assert_mdpi_clean_figure(fig, stem):
    """Fail fast if any figure-level or axes title was accidentally embedded."""
    problems = []

    if getattr(fig, "_suptitle", None) is not None:
        txt = fig._suptitle.get_text().strip()
        if txt:
            problems.append(f"suptitle={txt!r}")

    # fig.text(...) is reserved for figure-level text and is not allowed here.
    figure_level_text = [t.get_text().strip() for t in fig.texts if t.get_text().strip()]
    if figure_level_text:
        problems.append(f"figure_text={figure_level_text}")

    for k, ax in enumerate(fig.axes):
        for where, txt in [
            ("center", ax.get_title(loc="center")),
            ("left", ax.get_title(loc="left")),
            ("right", ax.get_title(loc="right")),
        ]:
            if str(txt).strip():
                problems.append(f"axes[{k}] title-{where}={txt!r}")

    if problems:
        raise RuntimeError(
            f"MDPI CLEAN CHECK FAILED for {stem}: " + "; ".join(problems)
        )


def save_pub_figure(fig, stem):
    assert_mdpi_clean_figure(fig, stem)
    for ext, kwargs in [
        ("png", {"dpi": 600}),
        ("pdf", {}),
        ("svg", {}),
    ]:
        path = OUT / f"{stem}.{ext}"
        fig.savefig(path, bbox_inches="tight", pad_inches=0.08, **kwargs)
        manifest.append({"figure": stem, "format": ext, "path": str(path)})
    print(f"MDPI clean check: PASS — {stem}")
    print(f"Saved: {stem} (PNG/PDF/SVG)")

def panel_label(ax, label):
    ax.text(
        -0.10, 1.04, label,
        transform=ax.transAxes,
        fontsize=13, fontweight="bold",
        va="top", ha="left",
        color=COLORS["black"]
    )

def minimal_grid(ax, axis="y"):
    ax.grid(axis=axis, alpha=0.16, linewidth=0.7)
    ax.set_axisbelow(True)

print("Publication visual system initialized.")


## Figure 1 — Pearson correlation matrix

In [ ]:

# ============================================================
# Figure 1. Correlation matrix
# ============================================================
corr = df[FEATURES].corr(method="pearson")

display_labels = [
    "Area", "Perimeter", "Major axis", "Minor axis", "Aspect ratio",
    "Convex area", "Equiv. diameter", "Extent", "Solidity",
    "Roundness", "Compactness"
]

fig, ax = plt.subplots(figsize=(8.8, 7.6))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")

ax.set_xticks(range(len(display_labels)))
ax.set_yticks(range(len(display_labels)))
ax.set_xticklabels(display_labels, rotation=42, ha="right")
ax.set_yticklabels(display_labels)

for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        v = corr.iloc[i, j]
        txt_color = "white" if abs(v) >= 0.58 else COLORS["black"]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=7.5, color=txt_color, fontweight="medium")

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.035)
cbar.set_label("Pearson correlation (r)")
cbar.outline.set_linewidth(0.6)

fig.tight_layout()
save_pub_figure(fig, "Figure1_Correlation_Matrix_PUBLICATION")
plt.show()


## Figure 2 — Descriptive LDA score projection

In [ ]:

# ============================================================
# Figure 2. Descriptive LDA projection
# ============================================================
X = df[FEATURES].to_numpy(dtype=float)
y = df[TARGET].astype(str).to_numpy()

Xs = StandardScaler().fit_transform(X)
lda = LinearDiscriminantAnalysis(n_components=2)
Z = lda.fit_transform(Xs, y)

expl = getattr(lda, "explained_variance_ratio_", np.array([np.nan, np.nan]))

fig, ax = plt.subplots(figsize=(8.5, 6.8))

for cls in CLASS_ORDER:
    m = y == cls
    ax.scatter(
        Z[m, 0], Z[m, 1],
        s=22, alpha=0.52,
        color=CLASS_COLORS[cls],
        edgecolors="white", linewidths=0.28,
        label=cls
    )
    cx, cy = Z[m, 0].mean(), Z[m, 1].mean()
    ax.scatter(
        cx, cy, s=175, marker="X",
        color=CLASS_COLORS[cls],
        edgecolor="black", linewidth=0.85, zorder=10
    )

xlab = "LD1"
ylab = "LD2"
if len(expl) >= 2 and np.isfinite(expl[:2]).all():
    xlab += f" ({100*expl[0]:.1f}% discriminant information)"
    ylab += f" ({100*expl[1]:.1f}%)"

ax.set_xlabel(xlab)
ax.set_ylabel(ylab)
minimal_grid(ax, "both")
ax.legend(ncol=2, loc="best")

fig.tight_layout()
save_pub_figure(fig, "Figure2_LDA_Projection_PUBLICATION")
plt.show()


## Figure 3 — Expanded FULL_11 benchmark

In [ ]:

# ============================================================
# Figure 3. Expanded FULL_11 benchmark
# ============================================================
m = metrics[metrics["feature_set"].astype(str).eq("FULL_11")].copy()

summary = (
    m.groupby("model", as_index=False)["f1_macro"]
     .agg(mean="mean", sd="std", n="count")
)
summary["display_model"] = summary["model"].map(clean_model_name)
summary = summary.sort_values("mean", ascending=True).reset_index(drop=True)

fig_h = max(6.8, 0.40 * len(summary) + 1.7)
fig, ax = plt.subplots(figsize=(9.4, fig_h))

top3 = set(summary.nlargest(3, "mean")["display_model"])
for i, row in summary.iterrows():
    name = row["display_model"]
    if name == "TabPFN":
        c, size = COLORS["vermillion"], 82
    elif name == "LDA-XGBoost":
        c, size = COLORS["blue"], 76
    elif name == "LDA-SVM":
        c, size = COLORS["green"], 76
    else:
        c, size = "#8C8C8C", 48

    ax.errorbar(row["mean"], i, xerr=row["sd"], fmt="none",
                ecolor=c, elinewidth=1.5, capsize=3.5, alpha=0.88)
    ax.scatter(row["mean"], i, s=size, color=c, edgecolor="white",
               linewidth=0.6, zorder=3)
    ax.text(row["mean"] + 0.0009, i, f'{row["mean"]:.4f}',
            va="center", fontsize=8.6,
            fontweight="bold" if name in top3 else "normal")

ax.set_yticks(range(len(summary)))
ax.set_yticklabels(summary["display_model"])
ax.set_xlabel("Mean outer-fold Macro-F1 ± SD")
minimal_grid(ax, "x")

xmin = max(0.94, (summary["mean"] - summary["sd"]).min() - 0.0015)
xmax = min(1.002, (summary["mean"] + summary["sd"]).max() + 0.0065)
ax.set_xlim(xmin, xmax)

fig.tight_layout()
save_pub_figure(fig, "Figure3_FULL11_Benchmark_PUBLICATION")
plt.show()

display(summary.sort_values("mean", ascending=False).head(10))


## Figure 4 — Mean row-normalized OOF confusion matrix

In [ ]:

# ============================================================
# Figure 4. LDA-XGBoost confusion matrix
# ============================================================
hx = hybrid_oof.copy()
hx["model_clean"] = hx["model"].map(clean_model_name)
ldax = hx[hx["model_clean"].eq("LDA-XGBoost")].copy()

if ldax.empty:
    raise ValueError("LDA-XGBoost predictions were not found in hybrid_oof_predictions.csv")

cms = []
for seed, g in ldax.groupby("seed"):
    cm = confusion_matrix(
        g["y_true"], g["y_pred"],
        labels=CLASS_ORDER,
        normalize="true"
    )
    cms.append(cm)

cm_mean = np.mean(cms, axis=0)

fig, ax = plt.subplots(figsize=(7.3, 6.5))
im = ax.imshow(cm_mean, cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_ORDER, rotation=32, ha="right")
ax.set_yticklabels(CLASS_ORDER)
ax.set_xlabel("Predicted cultivar")
ax.set_ylabel("True cultivar")

for i in range(4):
    for j in range(4):
        val = cm_mean[i, j]
        ax.text(
            j, i, f"{100*val:.1f}%",
            ha="center", va="center",
            fontsize=10.5,
            fontweight="bold" if i == j else "normal",
            color="white" if val > 0.54 else COLORS["black"]
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
cbar.set_label("Mean row-normalized proportion")

fig.tight_layout()
save_pub_figure(fig, "Figure4_Confusion_Matrix_LDAXGBoost_PUBLICATION")
plt.show()


## Figure 5 — Multiclass one-vs-rest ROC curves

In [ ]:

# ============================================================
# Figure 5. LDA-XGBoost ROC curves averaged across seeds
# ============================================================
grid = np.linspace(0, 1, 500)
seed_curves = {cls: [] for cls in CLASS_ORDER}
seed_aucs = {cls: [] for cls in CLASS_ORDER}

for seed, g in ldax.groupby("seed"):
    Y = label_binarize(g["y_true"].astype(str), classes=CLASS_ORDER)

    for j, cls in enumerate(CLASS_ORDER):
        fpr, tpr, _ = roc_curve(Y[:, j], g[f"prob_{cls}"].to_numpy())
        interp = np.interp(grid, fpr, tpr)
        interp[0], interp[-1] = 0.0, 1.0
        seed_curves[cls].append(interp)
        seed_aucs[cls].append(auc(fpr, tpr))

fig, ax = plt.subplots(figsize=(7.8, 6.6))

for cls in CLASS_ORDER:
    mean_tpr = np.mean(seed_curves[cls], axis=0)
    mean_auc = np.mean(seed_aucs[cls])
    ax.plot(
        grid, mean_tpr,
        linewidth=2.4,
        color=CLASS_COLORS[cls],
        label=f"{cls}  AUC={mean_auc:.3f}"
    )

ax.plot([0, 1], [0, 1], "--", color="#A6A6A6", linewidth=1.0, label="Chance")
ax.set_xlim(-0.01, 1.0)
ax.set_ylim(0.0, 1.01)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
minimal_grid(ax, "both")
ax.legend(loc="lower right")

fig.tight_layout()
save_pub_figure(fig, "Figure5_ROC_LDAXGBoost_PUBLICATION")
plt.show()


## Figure 6 — Raw vs cross-fitted temperature-scaled reliability

In [ ]:

# ============================================================
# Figure 6. Reliability diagrams
# Cross-fitted temperature scaling within each seed:
# fit T on 4 outer folds, apply to the held-out outer fold.
# ============================================================
def probs_to_logits(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0)
    return np.log(p)

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

def encode_y(values):
    mp = {c: i for i, c in enumerate(CLASS_ORDER)}
    return np.array([mp[v] for v in values], dtype=int)

def fit_temperature(p, y_idx):
    logits = probs_to_logits(p)
    def objective(T):
        T = max(float(T), 1e-5)
        q = softmax(logits / T)
        return -np.mean(np.log(q[np.arange(len(y_idx)), y_idx] + 1e-12))
    res = minimize_scalar(objective, bounds=(0.20, 5.0), method="bounded")
    return float(res.x)

def apply_temperature(p, T):
    return softmax(probs_to_logits(p) / float(T))

def crossfit_temperature(df_pred):
    df_pred = df_pred.reset_index(drop=True).copy()
    out = np.zeros((len(df_pred), len(CLASS_ORDER)), dtype=float)

    for seed, gs in df_pred.groupby("seed"):
        for fold in sorted(gs["outer_fold"].unique()):
            cal = gs[gs["outer_fold"] != fold]
            tst = gs[gs["outer_fold"] == fold]

            p_cal = cal[PROB_COLS].to_numpy(dtype=float)
            y_cal = encode_y(cal["y_true"].astype(str).to_numpy())
            T = fit_temperature(p_cal, y_cal)

            p_test = tst[PROB_COLS].to_numpy(dtype=float)
            scaled = apply_temperature(p_test, T)
            out[tst.index.to_numpy(), :] = scaled

    return out

def seed_reliability(df_pred, probs, n_bins=10):
    tmp = df_pred[["seed", "y_true"]].copy().reset_index(drop=True)
    y_idx = encode_y(tmp["y_true"].astype(str).to_numpy())
    pred_idx = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred_idx == y_idx).astype(float)

    edges = np.linspace(0, 1, n_bins + 1)
    rows = []
    for seed in sorted(tmp["seed"].unique()):
        mseed = tmp["seed"].to_numpy() == seed
        for b in range(n_bins):
            left, right = edges[b], edges[b+1]
            if b < n_bins - 1:
                mb = mseed & (conf >= left) & (conf < right)
            else:
                mb = mseed & (conf >= left) & (conf <= right)
            if mb.sum():
                rows.append({
                    "seed": seed,
                    "bin": b,
                    "mean_conf": conf[mb].mean(),
                    "accuracy": correct[mb].mean(),
                    "n": int(mb.sum()),
                })
    return pd.DataFrame(rows)

tab = tabpfn_oof[tabpfn_oof["feature_set"].astype(str).eq("FULL_11")].copy()
ldacal = ldax.copy()

models_for_cal = {
    "TabPFN": tab,
    "LDA-XGBoost": ldacal
}

fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.4), sharex=True, sharey=True)

for ax, (name, pred) in zip(axes, models_for_cal.items()):
    pred = pred.reset_index(drop=True)
    raw_probs = pred[PROB_COLS].to_numpy(dtype=float)
    scaled_probs = crossfit_temperature(pred)

    rel_raw = seed_reliability(pred, raw_probs)
    rel_cal = seed_reliability(pred, scaled_probs)

    def average_bins(rel):
        return (rel.groupby("bin", as_index=False)
                   .agg(mean_conf=("mean_conf", "mean"),
                        accuracy=("accuracy", "mean"),
                        n=("n", "sum")))

    rr = average_bins(rel_raw)
    rc = average_bins(rel_cal)

    # For publication display only, suppress very sparse bins (n < 10) because
    # their empirical accuracy is visually unstable. All bins remain included
    # in the ECE calculations reported in the numerical results.
    rr_display = rr[rr["n"] >= 10].copy()
    rc_display = rc[rc["n"] >= 10].copy()

    ax.plot([0.25, 1], [0.25, 1], "--", color="#A0A0A0", linewidth=1.1,
            label="Perfect calibration")
    ax.plot(rr_display["mean_conf"], rr_display["accuracy"], "-o",
            color=COLORS["orange"], linewidth=2.0, markersize=5.5,
            label="Raw")
    ax.plot(rc_display["mean_conf"], rc_display["accuracy"], "-o",
            color=COLORS["blue"], linewidth=2.0, markersize=5.5,
            label="Cross-fitted temperature")

    # Counts aggregate the three seed-specific OOF prediction sets and are
    # therefore not interpreted as independent biological grains.
    for _, r in rr_display.iterrows():
        ax.annotate(f'n={int(r["n"]):,}', (r["mean_conf"], r["accuracy"]),
                    textcoords="offset points", xytext=(-3, 9), ha="center",
                    fontsize=6.6, color=COLORS["orange"])
    for _, r in rc_display.iterrows():
        ax.annotate(f'n={int(r["n"]):,}', (r["mean_conf"], r["accuracy"]),
                    textcoords="offset points", xytext=(3, -13), ha="center",
                    fontsize=6.6, color=COLORS["blue"])

    ax.set_xlim(0.25, 1.005)
    ax.set_ylim(0.25, 1.005)
    ax.set_xlabel("Mean confidence")
    minimal_grid(ax, "both")
    ax.legend(loc="lower right", title=name, title_fontsize=9.5)
    ax.set_aspect("equal", adjustable="box")

axes[0].set_ylabel("Observed accuracy")
panel_label(axes[0], "A")
panel_label(axes[1], "B")

fig.tight_layout()
save_pub_figure(fig, "Figure6_Calibration_TabPFN_LDAXGBoost_PUBLICATION")
plt.show()


## Figure 7 — Accuracy–coverage trade-off

In [ ]:
# ============================================================
# Figure 7. Selective classification with fold-external thresholds
# ============================================================
sel = selective.copy()
sel["model_clean"] = sel["model"].map(clean_model_name)

keep_models = ["TabPFN", "LDA-XGBoost", "LDA-SVM"]
sel = sel[
    sel["feature_set"].astype(str).eq("FULL_11") &
    sel["model_clean"].isin(keep_models)
].copy()

if "calibration" in sel.columns:
    preferred = sel[
        sel["calibration"].astype(str).eq("temperature_crossfit_threshold_crossfit")
    ].copy()
    if not preferred.empty:
        sel = preferred

sum_sel = (
    sel.groupby(["model_clean", "target_coverage"], as_index=False)
       .agg(mean_cov=("actual_coverage", "mean"),
            sd_cov=("actual_coverage", "std"),
            mean_acc=("accuracy_accepted", "mean"),
            sd_acc=("accuracy_accepted", "std"))
)

fig, ax = plt.subplots(figsize=(8.3, 6.5))

for name in keep_models:
    g = sum_sel[sum_sel["model_clean"].eq(name)].sort_values("mean_cov")
    if g.empty:
        continue
    x = g["mean_cov"].to_numpy()
    y = g["mean_acc"].to_numpy()
    s = g["sd_acc"].fillna(0).to_numpy()
    c = MODEL_COLORS[name]

    ax.plot(x, y, "-o", color=c, linewidth=2.3, markersize=6, label=name)
    ax.fill_between(x, np.clip(y-s, 0, 1), np.clip(y+s, 0, 1),
                    color=c, alpha=0.10, linewidth=0)

ax.set_xlabel("Coverage (fraction of cases retained)")
ax.set_ylabel("Accuracy among retained cases")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlim(0.48, 1.01)
ax.set_ylim(0.965, 1.002)
minimal_grid(ax, "both")
ax.legend(loc="lower left")

fig.tight_layout()
save_pub_figure(fig, "Figure7_Accuracy_Coverage_PUBLICATION")
plt.show()

display(sum_sel[sum_sel["target_coverage"].isin([1.0, 0.9, 0.8])])


## Figure 8 — Leakage-safe diagnostic learning curves

In [ ]:

# ============================================================
# Figure 8. Learning curves
# ============================================================
lc = learning.copy()
lc["model_clean"] = lc["model"].map(clean_model_name)

models_lc = ["LDA-XGBoost", "SVM-RBF", "FT-Transformer"]
lc = lc[lc["model_clean"].isin(models_lc)].copy()

sum_lc = (
    lc.groupby(["model_clean", "train_fraction"], as_index=False)
      .agg(mean_f1=("test_macro_f1", "mean"),
           sd_f1=("test_macro_f1", "std"),
           mean_n=("n_train", "mean"))
)

fig, ax = plt.subplots(figsize=(8.3, 6.4))

for name in models_lc:
    g = sum_lc[sum_lc["model_clean"].eq(name)].sort_values("train_fraction")
    if g.empty:
        continue
    x = g["train_fraction"].to_numpy()
    y = g["mean_f1"].to_numpy()
    s = g["sd_f1"].fillna(0).to_numpy()
    c = MODEL_COLORS[name]

    ax.plot(x, y, "-o", color=c, linewidth=2.3, markersize=6, label=name)
    ax.fill_between(x, y-s, y+s, color=c, alpha=0.10, linewidth=0)

ax.set_xlabel("Training fraction")
ax.set_ylabel("Mean outer-test Macro-F1")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xticks(sorted(sum_lc["train_fraction"].unique()))
ax.set_ylim(max(0.94, sum_lc["mean_f1"].min() - 0.015), 0.995)
minimal_grid(ax, "both")
ax.legend(loc="lower right")

fig.tight_layout()
save_pub_figure(fig, "Figure8_Learning_Curves_PUBLICATION")
plt.show()

display(sum_lc)


## Final export package

In [ ]:
# ============================================================
# 5. Manifest, README, and ZIP package
# ============================================================
manifest_df = pd.DataFrame(manifest)
manifest_path = OUT / "NB10_Figure_Manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

readme_lines = [
    "NB10 R13 - MDPI Final Publication Figures",
    "Project: DRY_BEAN_HYBRID_Q1",
    "",
    "Generated figures:",
    "1. Correlation matrix",
    "2. Descriptive LDA score projection",
    "3. Expanded FULL_11 benchmark",
    "4. Mean row-normalized LDA-XGBoost OOF confusion matrix",
    "5. LDA-XGBoost multiclass OOF ROC curves",
    "6. TabPFN / LDA-XGBoost reliability diagrams with cross-fitted temperature scaling",
    "7. Accuracy-coverage curves",
    "8. Leakage-safe learning curves",
    "",
    "Important:",
    "- No model was retrained by NB10.",
    "- Original analytical outputs were not modified.",
    "- The raw dataset was not modified.",
    "- AspectRation was renamed to AspectRatio in memory only.",
    "- No figure title or descriptive subtitle is embedded inside the graphics.",
    "- Figure 3 model labels match manuscript terminology and error bars are not clipped.",
    "- Figure 6 displays populated-bin counts and includes low-confidence bins.",
    "- Figure 7 uses realized coverage from fold-external confidence-threshold selection generated by NB09 R2.",
    "- Figure titles/descriptions belong only in manuscript captions.",
    "- Automatic MDPI clean checks passed before export.",
    "- Figures were exported to PNG (600 dpi), PDF, and SVG.",
]
readme_path = OUT / "README_NB10_FIGURES.txt"
readme_path.write_text("\n".join(readme_lines), encoding="utf-8")

zip_base = str(OUT.parent / "NB10_PUBLICATION_FIGURES_MDPI_FINAL_R13")
zip_path = shutil.make_archive(zip_base, "zip", root_dir=OUT)

print("NB10 R13 finished successfully.")
print("Figures folder:", OUT)
print("Manifest:", manifest_path)
print("README:", readme_path)
print("ZIP:", zip_path)

display(manifest_df)

## Execution note

Run with **CPU**:

**Colab → Runtime → Change runtime type → Hardware accelerator: None**

Then use **Runtime → Run all**.

The notebook only reads saved outputs and regenerates publication figures. It does not retrain models.

Every exported figure is checked automatically to ensure that no figure title, suptitle, or figure-level descriptive text is embedded in the image.
